# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZainabFatima-hzf/ML-flyRank/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Primary task type: Classification** (binary), whose output probability doubles as a **ranking
score**.

The core question from Week 1 — "is this page about to decline, recover, or gain momentum?" —
sounds like three questions, but I'm anchoring the first concrete build on one clean binary
piece: **will this page decline next?** (yes/no). That matches the `framing-ml-problems` mapping
table directly: "Will this one decline / recover?" → classification, with a label from an
observed outcome. Recovery and momentum-gain are the same task type applied to a different slice
of pages (e.g. run the same model on currently-declining pages to predict recovery) — I'm not
duplicating the framing three times in Week 2, just naming that the pattern repeats.

The output isn't just "yes/no" though — I need the **predicted probability**, because the actual
deliverable (Week 1) is a *ranked* review queue, not a flat list of flags. So: classification
model → probability score → sorted into a ranked queue → top-K reviewed by a human. That's why
the success metric (Section 3) is precision@K, not plain accuracy.

In [1]:
print("Task type: Classification (binary) -> probability used as a ranking score")
print("Why not clustering: I have a specific outcome to predict (decline), not unlabeled groups to discover.")
print("Why not pure ranking/scoring: the score itself needs to come FROM a predicted outcome,")
print("not from a hand-built formula like the starter baseline_refresh_score.")


Task type: Classification (binary) -> probability used as a ranking score
Why not clustering: I have a specific outcome to predict (decline), not unlabeled groups to discover.
Why not pure ranking/scoring: the score itself needs to come FROM a predicted outcome,
not from a hand-built formula like the starter baseline_refresh_score.


## 2. Target or proxy

**What I'd predict (ideal target):** whether a page's traffic/engagement, measured over the
**next** 30 days, is meaningfully lower than the prior 90 days — a genuinely future-looking
outcome, built from `fact_content_daily_performance` in the warehouse, where I define my own
decision date, a prior feature window, and a later target window that doesn't overlap it.

**What I'm actually using right now (proxy, and I'm saying so out loud):** the starter CSV has no
daily granularity, so for this notebook I'm reusing the dataset's own
`is_declining_label = (trend_direction == "down")`. This is exactly the "beginner proxy label"
the lane guide itself warns about — it's a **defined rule** (last-30-days vs prior-30-days,
bucketed), not an **observed future outcome**. I confirmed in Week 1 that `trend_pct` correlates
1.0 with that same last-vs-prior comparison, so this label describes the recent past, not the
future.

**Why I'm using it anyway, honestly:** it's the only label the starter data supports, and it's
good enough to practice the *mechanics* of framing, building, and evaluating a classifier — as
long as I never claim it's a forecast. The moment I move to the warehouse, this proxy gets
replaced by a real forward-window label, and I'll re-run the leakage checks from scratch, since a
proxy this close to an existing rule is a leakage risk in itself if I'm not careful about which
columns I let the model see.

In [2]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("Proxy target 'is_declining_label' balance:")
print(df["is_declining_label"].value_counts(normalize=True).round(3))
print()
print("Reminder: trend_direction / trend_pct themselves can NEVER be model features —")
print("they ARE the label (or built the label). Confirmed via 1.0 correlation in Week 1.")


Proxy target 'is_declining_label' balance:
is_declining_label
1    0.542
0    0.458
Name: proportion, dtype: float64

Reminder: trend_direction / trend_pct themselves can NEVER be model features —
they ARE the label (or built the label). Confirmed via 1.0 correlation in Week 1.


## 3. Success metric

**Metric: precision@K**, where K = the number of pages a review team can actually look at in a
week (I'll use K=200 as a stand-in "team capacity" for this notebook — a real number I'd get from
FlyRank in practice). Precision@K asks: *of the top K pages my model ranks as most likely to
decline, what fraction genuinely carry the label?*

**Why this metric, not accuracy:** the action this supports is a short weekly review list, not a
blanket yes/no judgment on every page. Accuracy also hides class imbalance (54.2% of pages are
already "down," so a model could look "accurate" while being useless for prioritization).
Precision@K is the number a content lead would actually ask: "if I only have time for 200 pages,
how many of those 200 will actually be worth the visit?"

**The baseline it has to beat:** picking pages at random gives a precision equal to the base rate,
54.2%. A staleness-only ranking (oldest-updated pages first) — which sounds like a defensible rule
on its own — actually does *worse* than random at K=200, showing precision@K is sensitive to which
signal you rank by, and that a plausible-sounding single-variable rule can fail quietly. Any real
model has to beat both of these, not just "look reasonable."

In [3]:
df_sorted_random_proxy = df["is_declining_label"].mean()
print(f"Base rate / random-pick precision: {df_sorted_random_proxy:.3f}")

# Naive single-signal baseline: rank purely by staleness (days since last update), oldest first
K = 200
staleness_ranked = df.sort_values("days_since_last_update", ascending=False)
precision_at_k_staleness = staleness_ranked.head(K)["is_declining_label"].mean()
print(f"precision@{K} using a staleness-only ranking: {precision_at_k_staleness:.3f}")
print()
print("-> staleness alone actually underperforms random picking at this K.")
print("   'good' for my model means clearing 0.542 (base rate) by a real margin,")
print("   not just beating this one weak baseline.")


Base rate / random-pick precision: 0.542
precision@200 using a staleness-only ranking: 0.440

-> staleness alone actually underperforms random picking at this K.
   'good' for my model means clearing 0.542 (base rate) by a real margin,
   not just beating this one weak baseline.


## 4. The unit of analysis, as a real dataframe

**One row = one content page** (`content_id`), at the point in time this snapshot was taken. Below
is the actual slice of columns I'd hand a model for this task: identifiers (kept out of the model
itself, used only for joins/grouping), a small set of observable prior-state signals, and the
label column. This is deliberately not the full 44-column table — Section 5 shows the label
doesn't correlate strongly with any single one of these on its own, which is the real argument for
why this needs a model rather than a rule.

In [4]:
feature_cols = [
    "avg_position", "ctr", "engagement_rate", "scroll_rate",
    "content_age_days", "days_since_last_update", "word_count",
    "impressions_90d", "sessions_90d",
]
id_cols = ["content_id", "client_id"]  # grouping/joining only -- never model features
label_col = "is_declining_label"

lane_slice = df[id_cols + feature_cols + [label_col]]
print("Shape:", lane_slice.shape, "-> one row per content page")
lane_slice.head(5)


Shape: (30000, 12) -> one row per content page


,content_id,client_id,avg_position,ctr,engagement_rate,scroll_rate,content_age_days,days_since_last_update,word_count,impressions_90d,sessions_90d,is_declining_label
0,content_304f48230142,client_f369cb89fc,10.6,0.76,5.88,4.55,187,20,3221.0,3803,17,1
1,content_a1fb4e703a9e,client_4e07408562,20.3,0.05,0.00,10.00,445,25,2481.0,15320,9,1
2,content_9aa793d4d895,client_7f2253d7e2,36.5,0.09,0.00,28.57,141,20,3515.0,12581,11,1
3,content_331d6c4de07b,client_19581e27de,6.2,0.49,1.28,3.45,463,22,NaN,11751,78,0
4,content_d99b7a2d90ca,client_3fdba35f04,44.0,0.13,0.00,24.29,263,14,2803.0,19140,145,1


## 5. Why ML beats a fixed rule here

I tested this directly instead of asserting it. I tried the same style of rule FlyRank's own
baseline formula uses (`page_one_decay_risk`: top-10 position, page older than 180 days) as a
classifier for "will this page decline":

- **Precision: 0.518** — barely above the 54.2% base rate, i.e. almost the same as guessing.
- **Recall: 0.225** — it also misses over three-quarters of the pages that actually decline.

Then I checked every individual signal's correlation with the label — position, CTR, engagement
rate, scroll rate, content age, staleness, word count. **The strongest single correlation was
-0.164** (content age). None of them, alone, comes close to separating declining from non-declining
pages. That's the concrete version of "many signals, tangled, shifting over time" from the framing
skill: no single column, and no single hand-picked combination of two or three columns, cleanly
draws the line — but the *pattern* across all of them together plausibly does. That gap between
"any one rule is weak" and "the combination might not be" is exactly the space a model is meant to
fill, and exactly why I'm not just shipping an if-statement here.

In [5]:
rule = (df["avg_position"] > 0) & (df["avg_position"] <= 10) & (df["content_age_days"] >= 180)
flagged = rule.sum()
true_positives = ((rule) & (df["is_declining_label"] == 1)).sum()
precision = true_positives / flagged
recall = true_positives / df["is_declining_label"].sum()
print(f"Single-rule baseline (page_one_decay_risk style): flags {flagged} pages")
print(f"  precision = {precision:.3f}  (base rate is {df['is_declining_label'].mean():.3f})")
print(f"  recall    = {recall:.3f}")
print()

check_cols = ["avg_position", "engagement_rate", "scroll_rate", "ctr",
              "content_age_days", "days_since_last_update", "word_count"]
corrs = df[check_cols + ["is_declining_label"]].corr()["is_declining_label"].drop("is_declining_label")
print("Correlation of each single signal with the label:")
print(corrs.round(3).sort_values(key=abs, ascending=False))


Single-rule baseline (page_one_decay_risk style): flags 7076 pages
  precision = 0.518  (base rate is 0.542)
  recall    = 0.225

Correlation of each single signal with the label:
content_age_days         -0.164
word_count                0.090
days_since_last_update    0.081
ctr                      -0.062
avg_position             -0.029
engagement_rate          -0.013
scroll_rate              -0.003
Name: is_declining_label, dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.